# DiCo-NLI · AugB Haiku +1 — Kaggle T4（v2.2，Claude 改）

只改了 notebook，训练脚本 `train_augb.py` 不变。

**第一次（SMOKE）**：保持 `RUN_FORMAL = False`，在交互会话里 Run All。最后会打印结果包的位置和一个可点击的下载链接。

**正式 45 项**：把 `RUN_FORMAL` 改成 `True`，点右上角 **Save Version → Save & Run All (Commit)**，在后台跑（预计约 5 小时，不用开着页面）。跑完在这个 notebook 的版本页 **Output** 里下载 `augb_haiku_results.zip`。

**设置**：右侧 Add Input 挂上训练包数据集（`augb-haiku-kaggle`）和官方仓库数据集；Accelerator 选 **GPU T4 x1 或 x2**（只用第一张）；打开 Internet（下载模型）。

**中断后续跑**：把上一次版本的 Output（或下载的 `augb_haiku_results.zip` 上传成数据集）Add Input 挂上，再同样 Commit。第 7 格会自动把已完成的任务恢复进来，已完成的任务核对哈希后跳过。


In [ ]:
# 配置：第一次请保持 RUN_FORMAL=False
RUN_FORMAL = False
ONLY_SEED = None    # 42 / 43 / 44 / None(所有)
ONLY_FOLD = None    # 0..4 / None(所有)
ONLY_GROUP = "all"  # all / B0 / B1-Haiku / step+1

import os, sys, glob, pathlib, subprocess, zipfile, shutil, json
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
ROOT = pathlib.Path('/kaggle/working/augb_haiku')        # 训练包（从 /kaggle/input 复制过来）
OUT = pathlib.Path('/kaggle/working/augb_haiku_runs')    # 全部运行结果
RESULT_ZIP = pathlib.Path('/kaggle/working/augb_haiku_results.zip')

def pack(note=''):
    """把 OUT 打成 /kaggle/working/augb_haiku_results.zip（不含 .pt 权重），打印位置和下载链接。"""
    if not OUT.exists():
        print('还没有结果目录', OUT); return
    tmp = RESULT_ZIP.with_suffix('.tmp')
    n = 0
    with zipfile.ZipFile(tmp, 'w', zipfile.ZIP_DEFLATED) as z:
        for p in sorted(OUT.rglob('*')):
            if p.is_file() and p.suffix != '.pt':
                z.write(p, p.relative_to(OUT.parent)); n += 1
    tmp.replace(RESULT_ZIP)
    done = len(list(OUT.glob('tasks/**/done.json')))
    print(f'[结果包] {RESULT_ZIP}  {RESULT_ZIP.stat().st_size/1e6:.2f} MB，{n} 个文件；正式任务完成 {done}/45 {note}')
    print('下载：交互会话里点下面的链接，或右侧 Output 面板 → /kaggle/working → augb_haiku_results.zip → ⋮ → Download；Commit 版本在版本页 Output 里。')
    try:
        from IPython.display import FileLink, display
        display(FileLink(str(RESULT_ZIP.relative_to('/kaggle/working'))))
    except Exception:
        pass

print('配置完成。RUN_FORMAL =', RUN_FORMAL)


In [ ]:
# 安装缺失依赖（不重新安装 Kaggle 的 torch/CUDA）
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers>=4.45,<5','sentencepiece>=0.2','protobuf>=4.25','scikit-learn>=1.3'], check=True)

In [ ]:
# 找到训练包：Kaggle 上传 zip 后会自动解压成文件夹（路径形如 /kaggle/input/datasets/<用户>/<数据集>/）。
# 先找解压后的文件夹（同时含 train_augb.py 和 data/gate.json）；找不到再找原样的 zip。
hits = [p.parent.parent for p in pathlib.Path('/kaggle/input').glob('**/data/gate.json')
        if (p.parent.parent/'train_augb.py').is_file()
        and not (p.parent.parent.parent/'augb_haiku_runs').exists()]   # 排除挂进来的旧结果里那份复制品
if ROOT.exists():
    shutil.rmtree(ROOT)
if len(hits) == 1:
    shutil.copytree(hits[0], ROOT)   # /kaggle/input 只读，复制到 /kaggle/working 再用
    print('Training package folder:', hits[0])
elif len(hits) > 1:
    raise RuntimeError('找到多个训练包，请只挂一个：' + ', '.join(map(str, hits)))
else:
    files = sorted(glob.glob('/kaggle/input/**/AugB_Haiku_Kaggle_v2*.zip', recursive=True))
    if not files:
        raise FileNotFoundError('在 /kaggle/input 下没找到训练包（含 train_augb.py 和 data/gate.json 的文件夹，或 AugB_Haiku_Kaggle_v2.zip）。请在右侧 Add Input 挂上上传的数据集。')
    ROOT.mkdir(parents=True)
    with zipfile.ZipFile(files[0]) as z:
        for item in z.namelist():
            pp = pathlib.Path(item)
            if pp.is_absolute() or '..' in pp.parts:
                raise ValueError('Unsafe ZIP path: '+item)
        z.extractall(ROOT)
    print('Training package zip:', files[0])
assert (ROOT/'data/gate.json').is_file() and (ROOT/'train_augb.py').is_file()
print('Training package ready at:', ROOT, '（数据哈希由下一步的预检逐个核对）')


In [ ]:
# 找到官方仓库：Add Input 已解压目录 → /kaggle/working → 官方 zip → 在线 clone
def find_official():
    candidates = list(pathlib.Path('/kaggle/input').glob('**/final_data/train/dico_nli_train_track1_reference.csv'))
    candidates += list(pathlib.Path('/kaggle/working').glob('**/final_data/train/dico_nli_train_track1_reference.csv'))
    if candidates:
        return candidates[0].parents[2]
    archive_candidates = [q for q in glob.glob('/kaggle/input/**/*.zip', recursive=True)
                          if 'dico' in pathlib.Path(q).name.lower() or 'semeval' in pathlib.Path(q).name.lower()]
    for arch in archive_candidates:
        extracted = pathlib.Path('/kaggle/working/semeval_official_extracted')
        extracted.mkdir(exist_ok=True)
        with zipfile.ZipFile(arch) as z:
            z.extractall(extracted)
        candidates = list(extracted.glob('**/final_data/train/dico_nli_train_track1_reference.csv'))
        if candidates:
            return candidates[0].parents[2]
    clone = pathlib.Path('/kaggle/working/SemEval-2027-Task-2-DiCo-NLI')
    proc = subprocess.run(['git','clone','--depth','1',
             'https://github.com/ilopezgazpio/SemEval-2027-Task-2-DiCo-NLI.git',str(clone)],
              capture_output=True, text=True)
    if proc.returncode == 0 and (clone/'final_data/train/dico_nli_train_track1_reference.csv').exists():
        return clone
    raise FileNotFoundError('找不到官方训练 CSV；请 Add Input 官方仓库 ZIP，或在 Kaggle 开启 Internet。'+proc.stderr[-500:])
OFFICIAL = find_official()
print('Official repo:', OFFICIAL)

In [ ]:
# 第一关：全部数据验签 + 官方 3042 行 + 家族隔离 + 五折精确配额
BASE = [sys.executable, str(ROOT/'train_augb.py'), '--bundle',str(ROOT), '--official',str(OFFICIAL), '--out',str(OUT)]
subprocess.run(BASE+['--stage','preflight'], check=True)

In [ ]:
# 第二关：GPU SMOKE（每组两步，存取权重，内层验证）；跑完马上打结果包
subprocess.run(BASE+['--stage','smoke'], check=True)
for g in ('B0', 'B1-Haiku', 'stepplus1'):
    s = json.loads((OUT/'smoke'/g/'smoke.json').read_text())
    env = s['stable_environment']
    print(f"[SMOKE] {s['group']:9s} 更新 {s['updates']} 次  跳步 {s['skips']}  内层验证 WF1 {s['validation_wf1']:.4f}  用时 {s['elapsed_sec']:.0f} 秒  环境 {env}")
pack('（SMOKE）')


In [ ]:
# 续跑：如果 Add Input 挂了上一次的结果（文件夹 augb_haiku_runs 或 augb_haiku_results.zip），把已完成的任务恢复进 OUT
def find_previous():
    dirs = [p.parent.parent for p in pathlib.Path('/kaggle/input').glob('**/augb_haiku_runs/tasks/*') ]
    dirs = sorted({d for d in dirs})
    zips = sorted(glob.glob('/kaggle/input/**/augb_haiku_results.zip', recursive=True))
    return dirs, zips
dirs, zips = find_previous()
if dirs:
    zips = []   # 同一份旧结果里通常文件夹和 zip 都有，优先用文件夹
if len(dirs) > 1 or len(zips) > 1:
    raise RuntimeError('找到多份旧结果，请只挂一份：' + ', '.join(map(str, dirs + zips)))
if dirs:
    src = dirs[0]   # 就是旧的 augb_haiku_runs 文件夹
    print('从文件夹恢复：', src)
    shutil.copytree(src / 'tasks', OUT / 'tasks', dirs_exist_ok=True, ignore=shutil.ignore_patterns('*.pt'))
elif zips:
    print('从 zip 恢复：', zips[0])
    with zipfile.ZipFile(zips[0]) as z:
        for item in z.namelist():
            pp = pathlib.Path(item)
            if pp.is_absolute() or '..' in pp.parts or pp.parts[0] != 'augb_haiku_runs':
                raise ValueError('Unsafe ZIP path: ' + item)
        members = [m for m in z.namelist() if pathlib.PurePosixPath(m).parts[1:2] == ('tasks',)]
        z.extractall('/kaggle/working', members=members)
else:
    print('没有挂旧结果，从头开始。')
print('当前已完成任务：', len(list(OUT.glob('tasks/**/done.json'))), '/ 45（是否有效由训练脚本逐个核对哈希）')


In [ ]:
# 正式 45 项：只有 RUN_FORMAL=True 才跑；无论成功还是出错，最后都打结果包
if RUN_FORMAL:
    cmd = BASE+['--stage','full','--group',ONLY_GROUP]
    if ONLY_SEED is not None: cmd += ['--seed',str(ONLY_SEED)]
    if ONLY_FOLD is not None: cmd += ['--fold',str(ONLY_FOLD)]
    try:
        subprocess.run(cmd, check=True)
    finally:
        pack('（正式）')
        shutil.rmtree(ROOT, ignore_errors=True)   # 训练包复制品不放进 Output，续跑时不会和原数据集重复
    summary = OUT/'summary.json'
    print('summary.json 已生成' if summary.exists() else '没有 summary.json（只跑了部分任务，或中途出错）')
else:
    print('只跑了 SMOKE，45 项正式任务没有启动。把 RUN_FORMAL 改成 True，再 Save Version → Save & Run All (Commit)。')
